# MLP erstellen (Betonstärkenprädiktion)

In [2]:
import kagglehub
import pandas
import matplotlib.pyplot as plt
import numpy

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

import torch
from torch import nn

# 1. Daten einlesen
path = kagglehub.dataset_download("mchilamwar/predict-concrete-strength")
df = pandas.read_csv(f"{path}/ConcreteStrengthData.csv")
df = df.rename( {"CementComponent " : "CementComponent"}, axis="columns" )

# 2. Was ist Input? Was ist Output?
#    Train-Test-Split
target = "Strength"
x = df.drop( target, axis="columns" )
y = df[ [target] ]
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2,
                                                    random_state=42)

# 3. Skalierung der Daten (Achtung vor Data Leakage!)
x_scaler = RobustScaler()
y_scaler = RobustScaler()
x_train_scaled = x_scaler.fit_transform(x_train)
y_train_scaled = y_scaler.fit_transform(y_train)

# 4. Daten als Tensoren "verpacken"
x_train_scaled_tensor = torch.tensor(x_train_scaled, dtype=torch.float32)
y_train_scaled_tensor = torch.tensor(y_train_scaled, dtype=torch.float32)

# 5. MLP bauen
N1 = 128
N2 = 64
nr_inputs = len(x_train.columns)
nr_outputs = len(y_train.columns)
model = nn.Sequential(
            nn.Linear(nr_inputs, N1),
            nn.ReLU(),

            nn.Linear(N1, N2),
            nn.ReLU(),
    
            nn.Linear(N2, nr_outputs)
)

# 6. Jetzt das MLP trainieren
model.train()
loss_function = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
for epoch in range(1000):
    predictions = model(x_train_scaled_tensor)
    loss = loss_function(predictions, y_train_scaled_tensor)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

# 7. Jetzt das MLP testen
model.eval()
x_test_scaled = x_scaler.transform(x_test)
x_test_scaled_tensor = torch.tensor(x_test_scaled, dtype=torch.float32)
with torch.no_grad():
    predictions_test_scaled_tensor = model(x_test_scaled_tensor)
predictions_test = y_scaler.inverse_transform( predictions_test_scaled_tensor )

# 8. Wie gut ist das MLP?
y_pred = predictions_test.flatten()
y_true = y_test[target].values
MAE = mean_absolute_error(y_true, y_pred)
MAPE = mean_absolute_percentage_error(y_true, y_pred)*100
print(f"MAE={MAE:.2f} [Pascal], MAPE={MAPE:.2f}%")

MAE=3.23 [Pascal], MAPE=10.07%


# MLP inklusive Skalierung als ONNX exportieren

Zuerst die Trainingszelle oben ausführen. Wir speichern die gesamte Berechnung in **einer Datei**:

**Rohdaten → Eingabeskalierung → MLP → Rückskalierung → Betonfestigkeit**

Für die hier verwendeten `RobustScaler` sind das einfache Rechenoperationen:
- Eingaben skalieren: `(x - x_scaler.center_) / x_scaler.scale_`
- Ausgabe zurückskalieren: `y_scaled * y_scaler.scale_ + y_scaler.center_`

Die bereits gelernten Skalierungswerte werden als Konstanten mit exportiert. Es wird nichts erneut trainiert oder angepasst.

Benötigt werden `onnx` und `onnxruntime` (in `env_teaching` bereits vorhanden). Falls sie in einer anderen Umgebung fehlen: `%pip install onnx onnxruntime`.

In [2]:
class MLPMitSkalierung(nn.Module):
    def __init__(self, mlp, x_scaler, y_scaler):
        super().__init__()
        self.mlp = mlp
        # Buffer sind feste Werte, keine trainierbaren Parameter.
        self.register_buffer("x_center", torch.tensor(x_scaler.center_, dtype=torch.float32))
        self.register_buffer("x_scale", torch.tensor(x_scaler.scale_, dtype=torch.float32))
        self.register_buffer("y_center", torch.tensor(y_scaler.center_, dtype=torch.float32))
        self.register_buffer("y_scale", torch.tensor(y_scaler.scale_, dtype=torch.float32))

    def forward(self, x):
        x_scaled = (x - self.x_center) / self.x_scale
        y_scaled = self.mlp(x_scaled)
        return y_scaled * self.y_scale + self.y_center


export_model = MLPMitSkalierung(model, x_scaler, y_scaler).eval()

In [3]:
type(export_model)

__main__.MLPMitSkalierung

In [9]:
%pip install onnxscript

  Using cached onnxscript-0.7.2-py3-none-any.whl.metadata (13 kB)
Using cached onnxscript-0.7.2-py3-none-any.whl (754 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [onnxscript]2 [onnxscript]
Note: you may need to restart the kernel to use updated packages.


In [10]:
# Zwei Beispielzeilen für den Export mit variabler Batchgröße
beispiel = torch.tensor(
    x_train.iloc[:2].to_numpy(), dtype=torch.float32
)

torch.onnx.export(
    export_model,
    (beispiel,),
    "beton_mlp.onnx",
    input_names=["eingaben"],
    output_names=["betonfestigkeit"],
    dynamic_shapes={"x": {0: torch.export.Dim("anzahl")}},
    opset_version=18,
    dynamo=True,
    external_data=False,
)

[torch.onnx] Obtain model graph for `MLPMitSkalierung([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `MLPMitSkalierung([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


ONNXProgram(
    model=
        <
            ir_version=10,
            opset_imports={'': 18},
            producer_name='pytorch',
            producer_version='2.9.1+cu128',
            domain=None,
            model_version=None,
        >
        graph(
            name=main_graph,
            inputs=(
                %"eingaben"<FLOAT,[anzahl,8]>
            ),
            outputs=(
                %"betonfestigkeit"<FLOAT,[anzahl,1]>
            ),
            initializers=(
                %"mlp.0.weight"<FLOAT,[128,8]>{TorchTensor(...)},
                %"mlp.0.bias"<FLOAT,[128]>{TorchTensor(...)},
                %"mlp.2.weight"<FLOAT,[64,128]>{TorchTensor(...)},
                %"mlp.2.bias"<FLOAT,[64]>{TorchTensor(...)},
                %"mlp.4.weight"<FLOAT,[1,64]>{TorchTensor(...)},
                %"mlp.4.bias"<FLOAT,[1]>{TorchTensor<FLOAT,[1]>(Parameter containing: tensor([0.1139], requires_grad=True), name='mlp.4.bias')},
                %"x_center"<FLOAT,[8]>{TorchTe

"x" bezeichnet das Argument von forward(self, x). external_data=False sorgt dafür, dass die Gewichte direkt in unserer einzelnen ONNX-Datei gespeichert werden.

In [11]:
!ls -la

total 84
drwxrwxr-x  2 juebrauer juebrauer  4096 Oct 21 08:08 .
drwxrwxr-x 77 juebrauer juebrauer  4096 Oct 21 08:23 ..
-rw-rw-r--  1 juebrauer juebrauer 49148 Oct 21 08:25 beton_mlp.onnx
-rw-rw-r--  1 juebrauer juebrauer 24723 Oct 21 08:27 onnx_beispiele.ipynb


# ONNX-Datei laden und neue Daten vorhersagen

**Dieser Abschnitt funktioniert auch nach einem Kernel-Neustart**, sobald `beton_mlp.onnx` im Arbeitsverzeichnis liegt. Für die Vorhersage benötigen wir nur ONNX Runtime, NumPy und hier Pandas zur Tabellenanzeige.

Die zwei Zeilen unten sind beispielhafte Betonmischungen, deren Werte du ändern kannst. Wichtig: **dieselbe Spaltenreihenfolge wie beim Training**, `float32` und eine zweidimensionale Eingabe `(Anzahl Zeilen, 8)` – auch bei nur einer Mischung. Die Daten werden **unskaliert** übergeben; beide Skalierungen stecken bereits in der Datei.

In [12]:
import numpy as np
import pandas as pd
import onnxruntime as ort

session = ort.InferenceSession("beton_mlp.onnx", providers=["CPUExecutionProvider"])

# Reihenfolge wie beim Training: sieben Mischungsmerkmale, dann Alter in Tagen.
neue_daten = pd.DataFrame(
    [
        [300.0, 50.0, 20.0, 180.0, 5.0, 1000.0, 750.0, 28.0],
        [350.0, 40.0, 30.0, 170.0, 6.0, 1050.0, 700.0, 56.0],
    ],
    columns=[
        "CementComponent", "BlastFurnaceSlag", "FlyAshComponent",
        "WaterComponent", "SuperplasticizerComponent",
        "CoarseAggregateComponent", "FineAggregateComponent", "AgeInDays",
    ],
)

# run() liefert eine Liste der Modellausgaben; unser Modell hat genau eine.
eingaben = neue_daten.to_numpy(dtype=np.float32)
praediktionen = session.run(None, {"eingaben": eingaben})[0]

ergebnis = neue_daten.copy()
ergebnis["Vorhergesagte Betonfestigkeit"] = praediktionen.ravel()
ergebnis

,CementComponent,BlastFurnaceSlag,FlyAshComponent,WaterComponent,SuperplasticizerComponent,CoarseAggregateComponent,FineAggregateComponent,AgeInDays,Vorhergesagte Betonfestigkeit
0,300.0,50.0,20.0,180.0,5.0,1000.0,750.0,28.0,36.761852
1,350.0,40.0,30.0,170.0,6.0,1050.0,700.0,56.0,52.509460


Die Vorhersagen sind bereits in der ursprünglichen Einheit der Zielspalte `Strength`. Für diese neuen Mischungen sind keine tatsächlichen Zielwerte bekannt; die Tabelle zeigt Modellvorhersagen.

Dokumentation: [ONNX Runtime – Python API](https://onnxruntime.ai/docs/api/python/api_summary.html).

# Optional: Stimmen PyTorch und ONNX überein?

Diese Kontrollzelle benötigt zusätzlich die Trainingszelle im selben Kernel. Wir vergleichen auf den Testdaten die bisherige Berechnung (Scikit-learn-Skalierer + PyTorch) mit der ONNX-Datei. Kleine Abweichungen durch `float32` und unterschiedliche Rechenwege sind normal.

In [13]:
onnx_test = session.run(None, {"eingaben": x_test.to_numpy(dtype=np.float32)})[0]

max_abweichung = np.max(np.abs(predictions_test - onnx_test))
print(f"Maximale absolute Abweichung: {max_abweichung:.6f}")
np.testing.assert_allclose(onnx_test, predictions_test, rtol=1e-4, atol=1e-4)
print("PyTorch und ONNX liefern praktisch gleiche Vorhersagen.")

Maximale absolute Abweichung: 0.000021
PyTorch und ONNX liefern praktisch gleiche Vorhersagen.


atol erlaubt eine kleine absolute Abweichung. Das ist besonders bei Werten nahe null wichtig.

rtol erlaubt eine Abweichung proportional zur Größe des Referenzwerts.

# Dasselbe Problem mit einem Random Forest

Wir verwenden dieselben Trainings- und Testdaten (`x_train`, `x_test`, `y_train`, `y_test`) aus der ersten Trainingszelle. Ein `RandomForestRegressor` mittelt die Vorhersagen vieler Entscheidungsbäume. Hier benötigen wir **keine Skalierung** der Eingaben oder Zielwerte.

Für den ONNX-Export von Scikit-learn-Modellen verwenden wir `skl2onnx`. Falls das Paket fehlt: `%pip install skl2onnx`.

## 1. Trainieren und testen

In [14]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(x_train, y_train["Strength"])  # Eine Zielspalte als eindimensionaler Vektor

rf_test = rf.predict(x_test)
print(f"MAE: {mean_absolute_error(y_test['Strength'], rf_test):.2f}")
print(f"MAPE: {mean_absolute_percentage_error(y_test['Strength'], rf_test):.2%}")

MAE: 3.74
MAPE: 12.32%


## 2. Random Forest als ONNX exportieren

Eine Beispielzeile legt Eingabetyp und Anzahl der Merkmale fest. Das exportierte Modell kann anschließend beliebig viele Zeilen verarbeiten. Alle Bäume werden in `beton_rf.onnx` gespeichert.

Dokumentation: [sklearn-onnx – Einführung](https://onnx.ai/sklearn-onnx/introduction.html).

In [15]:
import numpy as np
from skl2onnx import to_onnx

rf_onnx = to_onnx(
    rf,
    x_train.iloc[:1].to_numpy(dtype=np.float32),
    target_opset=18,
)
with open("beton_rf.onnx", "wb") as datei:
    datei.write(rf_onnx.SerializeToString())

print("Gespeichert: beton_rf.onnx")

Gespeichert: beton_rf.onnx


In [16]:
type(rf_onnx)

onnx.onnx_ml_pb2.ModelProto

## 3. ONNX-Datei laden und anwenden

Diese Zelle funktioniert **auch nach einem Kernel-Neustart**, wenn `beton_rf.onnx` im Arbeitsverzeichnis liegt. Der trainierte Python-Random-Forest wird dafür nicht benötigt.

Wir übergeben unskalierte Daten als `float32`, in derselben Spaltenreihenfolge wie beim Training. Die Ausgabe ist bereits in der ursprünglichen Einheit von `Strength`.

In [19]:
import numpy as np
import pandas as pd
import onnxruntime as ort

rf_session = ort.InferenceSession("beton_rf.onnx", providers=["CPUExecutionProvider"])

rf_neue_daten = pd.DataFrame(
    [
        [300.0, 50.0, 20.0, 180.0, 5.0, 1000.0, 750.0, 28.0],
        [350.0, 40.0, 30.0, 170.0, 6.0, 1050.0, 700.0, 56.0],
    ],
    columns=[
        "CementComponent", "BlastFurnaceSlag", "FlyAshComponent",
        "WaterComponent", "SuperplasticizerComponent",
        "CoarseAggregateComponent", "FineAggregateComponent", "AgeInDays",
    ],
)

input_name = rf_session.get_inputs()[0].name
rf_eingaben = rf_neue_daten.to_numpy(dtype=np.float32)
rf_praediktionen = rf_session.run(None, {input_name: rf_eingaben})[0]

rf_ergebnis = rf_neue_daten.copy()
rf_ergebnis["Vorhergesagte Betonfestigkeit"] = rf_praediktionen.ravel()
rf_ergebnis

,CementComponent,BlastFurnaceSlag,FlyAshComponent,WaterComponent,SuperplasticizerComponent,CoarseAggregateComponent,FineAggregateComponent,AgeInDays,Vorhergesagte Betonfestigkeit
0,300.0,50.0,20.0,180.0,5.0,1000.0,750.0,28.0,41.457813
1,350.0,40.0,30.0,170.0,6.0,1050.0,700.0,56.0,54.279018


In [20]:
input_name

'X'

In [21]:
rf_eingaben

array([[ 300.,   50.,   20.,  180.,    5., 1000.,  750.,   28.],
       [ 350.,   40.,   30.,  170.,    6., 1050.,  700.,   56.]],
      dtype=float32)

## Optional: Scikit-learn und ONNX vergleichen

Diese Kontrolle benötigt die RF-Trainingszelle im selben Kernel. `ravel()` bringt die ONNX-Ausgabe in dieselbe eindimensionale Form wie `rf.predict()`. Kleine Rundungsabweichungen sind normal.

In [22]:
rf_onnx_test = rf_session.run(
    None, {input_name: x_test.to_numpy(dtype=np.float32)}
)[0].ravel()

print(f"Maximale absolute Abweichung: {np.max(np.abs(rf_onnx_test - rf_test)):.6f}")
np.testing.assert_allclose(rf_onnx_test, rf_test, rtol=1e-4, atol=1e-4)
print("Scikit-learn und ONNX liefern praktisch gleiche Vorhersagen.")

Maximale absolute Abweichung: 0.000044
Scikit-learn und ONNX liefern praktisch gleiche Vorhersagen.


# Betonfestigkeit mit einem kleinen Transformer

Ein Transformer braucht hier keine Zeitreihe: Wir machen aus **jedem der acht Merkmale ein Token**. Jedes Merkmal erhält eigene lernbare Gewichte und einen Bias, damit etwa Zement und Alter unterscheidbar bleiben. Self-Attention verknüpft die Merkmale **innerhalb einer Mischung**, nicht verschiedene Datenzeilen. Anschließend mitteln wir die acht Token und sagen eine Zahl voraus.

Das ist ein kleines Lehrbeispiel, inspiriert von [FT-Transformer](https://arxiv.org/abs/2106.11959), keine vollständige Implementierung dieses Modells. Für diese kleine Tabelle ist ein Vorteil gegenüber MLP oder Baummodellen nicht selbstverständlich.

## 1. Daten für einen kontrollierten Vergleich

Benötigt werden `x_train`, `x_test`, `y_train`, `y_test` aus der ersten Notebook-Zelle. Wir teilen **nur die Trainingsdaten** nochmals in Training und Validierung. Die neuen Skalierer lernen ausschließlich aus diesem kleineren Trainingssatz. Der bisherige Testdatensatz bleibt unverändert. Die bisherigen MLP- und RF-Modelle werden nicht überschrieben.

In [3]:
import copy
import time
import numpy as np
import pandas as pd
import torch
from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

x_fit, x_val, y_fit, y_val = train_test_split(
    x_train, y_train, test_size=0.2, random_state=42
)
vergleich_x_scaler = RobustScaler().fit(x_fit)
vergleich_y_scaler = RobustScaler().fit(y_fit)

def tensor(werte):
    return torch.tensor(werte, dtype=torch.float32)

X_fit = tensor(vergleich_x_scaler.transform(x_fit))
X_val = tensor(vergleich_x_scaler.transform(x_val))
X_test = tensor(vergleich_x_scaler.transform(x_test))
Y_fit = tensor(vergleich_y_scaler.transform(y_fit))
Y_val = tensor(vergleich_y_scaler.transform(y_val))
print(f"Training: {len(x_fit)}, Validierung: {len(x_val)}, Test: {len(x_test)}")

Training: 659, Validierung: 165, Test: 206


## 2. Architektur

Formen der Tensoren: `(Mischungen, 8)` → `(Mischungen, 8, 16)` → `(Mischungen, 16)` → `(Mischungen, 1)`.

`batch_first=True` bedeutet, dass die erste Dimension die Mischungen zählt. Vier Attention-Heads betrachten verschiedene gelernte Beziehungen zwischen den Merkmalen. Für die festen Merkmalsidentitäten genügen hier die eigenen Token-Gewichte; eine zeitliche Positionskodierung benötigen wir nicht.

[PyTorch-Dokumentation: TransformerEncoderLayer](https://docs.pytorch.org/docs/stable/generated/torch.nn.TransformerEncoderLayer.html)

In [5]:
class BetonTransformer(nn.Module):
    def __init__(self, nr_merkmale, breite=16):
        super().__init__()
        self.token_gewicht = nn.Parameter(torch.randn(nr_merkmale, breite) * 0.1)
        self.token_bias = nn.Parameter(torch.randn(nr_merkmale, breite) * 0.1)
        self.attention = nn.TransformerEncoderLayer(
            d_model=breite, nhead=4, dim_feedforward=32,
            dropout=0.1, batch_first=True,
        )
        self.ausgabe = nn.Linear(breite, 1)

    def forward(self, x):
        tokens = x.unsqueeze(-1) * self.token_gewicht + self.token_bias
        tokens = self.attention(tokens)
        return self.ausgabe(tokens.mean(dim=1))


def neues_mlp():
    # Dieselbe Architektur wie das MLP am Anfang des Notebooks.
    return nn.Sequential(
        nn.Linear(X_fit.shape[1], 128), nn.ReLU(),
        nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 1),
    )

## 3. Trainieren und vergleichen

Beide Modelle erhalten dieselben Daten, Adam mit derselben Lernrate und höchstens 500 Epochen. Wir speichern jeweils den Zustand mit dem kleinsten Validierungsfehler (MSE) und stoppen nach 60 Epochen ohne Verbesserung. Drei Zufallsstarts zeigen, wie stark das Ergebnis von Initialisierung und Training abhängt.

Das ist ein kompakter Vergleich mit festgelegten Einstellungen, **keine Hyperparameteroptimierung**. Die früheren Notebook-Ergebnisse sind nicht direkt gleichzusetzen: Dort wurde ohne diesen zusätzlichen Validierungssplit trainiert. Laufzeiten hängen vom Rechner ab; hier laufen beide Modelle auf der CPU.

In [ ]:
def trainieren(netz):
    optimizer = torch.optim.Adam(netz.parameters(), lr=0.003)
    loss_fn = nn.MSELoss()
    bester_loss = float("inf")
    warten = 0
    start = time.perf_counter()

    for epoche in range(500):
        netz.train()
        optimizer.zero_grad()
        loss = loss_fn(netz(X_fit), Y_fit)
        loss.backward()
        optimizer.step()

        netz.eval()
        with torch.no_grad():
            val_loss = loss_fn(netz(X_val), Y_val).item()
        if val_loss < bester_loss:
            bester_loss = val_loss
            bester_zustand = copy.deepcopy(netz.state_dict())
            beste_epoche = epoche + 1
            warten = 0
        else:
            warten += 1
        if warten >= 60:
            break

    netz.load_state_dict(bester_zustand)
    netz.eval()
    return beste_epoche, time.perf_counter() - start


resultate = []
vergleich_modelle = {}
for seed in [0, 1, 2]:
    for name in ["MLP", "Transformer"]:
        torch.manual_seed(seed)
        netz = neues_mlp() if name == "MLP" else BetonTransformer(X_fit.shape[1])
        beste_epoche, sekunden = trainieren(netz)
        with torch.no_grad():
            vorhersage = vergleich_y_scaler.inverse_transform(netz(X_test).numpy()).ravel()
        resultate.append({
            "Modell": name, "Seed": seed,
            "MAE": mean_absolute_error(y_test["Strength"], vorhersage),
            "MAPE (%)": 100 * mean_absolute_percentage_error(y_test["Strength"], vorhersage),
            "Sekunden": sekunden, "Beste Epoche": beste_epoche,
            "Parameter": sum(p.numel() for p in netz.parameters()),
        })
        vergleich_modelle[(name, seed)] = netz
        print(f"{name}, Seed {seed}: MAE={resultate[-1]['MAE']:.2f}, {sekunden:.1f} s")

vergleich = pd.DataFrame(resultate)
vergleich.groupby("Modell")[["MAE", "MAPE (%)", "Sekunden"]].agg(["mean", "std"]).round(3)

MLP, Seed 0: MAE=3.31, 2.1 s
Transformer, Seed 0: MAE=4.24, 5.5 s
MLP, Seed 1: MAE=3.49, 2.3 s
Transformer, Seed 1: MAE=4.17, 6.1 s
MLP, Seed 2: MAE=3.74, 1.4 s
Transformer, Seed 2: MAE=4.19, 5.6 s


MAE        MAPE (%)        Sekunden       
              mean    std     mean    std     mean    std
Modell                                                   
MLP          3.513  0.220   11.304  0.174    1.942  0.473
Transformer  4.200  0.036   12.723  0.191    5.749  0.296

In [7]:
vergleich

,Modell,Seed,MAE,MAPE (%),Sekunden,Beste Epoche,Parameter
0,MLP,0,3.305153,11.142733,2.064269,494,9473
1,Transformer,0,4.240023,12.898861,5.533235,419,2497
2,MLP,1,3.491599,11.489251,2.341709,493,9473
3,Transformer,1,4.171576,12.519363,6.086644,487,2497
4,MLP,2,3.742612,11.280062,1.419007,254,9473
5,Transformer,2,4.187979,12.750772,5.627932,476,2497


## 4. Vorhersagen ansehen

Wir verwenden vorab festgelegt den Transformer mit Seed 0, nicht den mit dem besten Testfehler. Die ersten fünf Testmischungen waren für das Training unbekannt. Für eigene Mischungen kann `x_test.iloc[:5]` durch einen DataFrame mit denselben acht Spalten in derselben Reihenfolge ersetzt werden.

In [8]:
transformer = vergleich_modelle[("Transformer", 0)]
beispiele = x_test.iloc[:5]
with torch.no_grad():
    pred_scaled = transformer(tensor(vergleich_x_scaler.transform(beispiele)))
pred = vergleich_y_scaler.inverse_transform(pred_scaled.numpy()).ravel()

pd.DataFrame({
    "Tatsächliche Festigkeit": y_test["Strength"].iloc[:5].to_numpy(),
    "Transformer-Vorhersage": pred,
}, index=beispiele.index)

,Tatsächliche Festigkeit,Transformer-Vorhersage
31,52.91,52.423069
109,55.90,43.255070
136,74.50,74.010849
88,35.30,33.784332
918,10.54,16.025667


## Macht der Transformer hier Sinn?

**Als Lehrbeispiel ja:** Attention kann Wechselwirkungen zwischen den Merkmalen modellieren. Ein MLP kann solche Wechselwirkungen ebenfalls lernen. Die acht Spalten bilden keine natürliche zeitliche oder sprachliche Sequenz; es gibt hier auch kein vortrainiertes Sprachmodell und keinen automatischen Wissensvorsprung.

**Als praktische Modellwahl entscheidet der Vergleich:** Ein kleinerer mittlerer Test-MAE ist besser; berücksichtige auch die Streuung über die Seeds und die Laufzeit. Bei wenigen Daten kann die zusätzliche Flexibilität überanpassen. Ein Random Forest oder ein MLP bleibt eine sinnvolle Ausgangsbasis.

Die drei Seeds verwenden denselben Datensplit: Sie messen Trainingsschwankungen, nicht die Unsicherheit über unterschiedliche Datensätze. Ein kleiner Abstand beweist daher keinen allgemeinen Vorteil. Für eine belastbare Modellwahl wären mehrere Splits beziehungsweise Kreuzvalidierung und eine separate Hyperparameterabstimmung nötig. Nach Sichtung dieser Testwerte sollte der Testdatensatz nicht zur weiteren Optimierung verwendet werden.

### Ergebnis des ausgeführten Beispiels

Getestet in `env_teaching` mit dem lokal vorhandenen Datensatz (1.030 Zeilen), auf der CPU mit zwei PyTorch-Rechenthreads. Mittelwert ± Stichprobenstandardabweichung über drei Seeds:

| Modell | Test-MAE | Test-MAPE | Trainingszeit | Parameter |
|---|---:|---:|---:|---:|
| MLP | 3,486 ± 0,207 | 11,177 % | 0,372 s | 9.473 |
| Kleiner Transformer | 4,202 ± 0,039 | 12,739 % | 4,670 s | 2.497 |

**In diesem Versuch bringt der Transformer keinen Vorteil:** Sein mittlerer Test-MAE ist etwa 21 % höher und das Training dauert etwa 12,6-mal so lange. Weniger Parameter bedeuten also nicht automatisch weniger Rechenaufwand. Die Architekturen sind nicht auf gleiche Parameterzahl abgestimmt; beide erhalten denselben Trainingsablauf, aber keine individuell optimierten Einstellungen. Die gemessenen Zeiten schließen das Training und die Validierung ein, nicht die abschließende Testvorhersage.

Für den Kurs ist das ein nützliches Gegenbeispiel zu „modernere Architektur = automatisch besser“. Für diese Aufgabe würde ich anhand dieses Versuchs beim MLP beziehungsweise beim Vergleich mit Baummodellen bleiben. Andere Einstellungen oder Datensplits können das Ergebnis verändern. Bei erneutem Ausführen sind die dann erzeugten Tabellen maßgeblich.
